# Bronze Layer — TPC-A Ingestion

Ingests the eight TPC-A source CSVs from the raw volume (`/Volumes/skyledger_airlines/bronze/raw`) into Bronze Delta tables, preserving all source columns and grain, and appends ingestion metadata (`_ingestion_timestamp`, `_source_file`, `_pipeline_run_id`). Ingestion is idempotent — full overwrite, so rerunning produces the same result as running once.

In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Catalog widget — keeps this notebook consistent with 00_setup.
dbutils.widgets.text("CATALOG", "skyLedger_airlines")
CATALOG = dbutils.widgets.get("CATALOG")

BASE_PATH = f"/Volumes/{CATALOG.lower()}/bronze/raw"

# One run ID per pipeline execution — stamped on every row for traceability.
from datetime import datetime, timezone

RUN_ID = f"tpca_ingest_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"
INGESTED_AT = datetime.now(timezone.utc)

# (table_name, source_folder, source_file, business meaning)
SOURCES = [
    ("passenger",       "passenger",      "passenger.csv",      "Passenger master — demographics and travel purpose"),
    ("booking",         "booking",        "booking.csv",        "Passenger bookings with dates, status, and total price"),
    ("flight_booking",  "flight booking", "flight_booking.csv", "Booking line items — one row per flight segment, with seats, ticket price, discounts, taxes, and travel dates"),
    ("route",           "route",          "route.csv",          "Flight route master — origin/destination pairs, aircraft type, and base fare"),
    ("airline_route",    "airline route",  "airline_route.csv", "Airline-to-route operational relationships, available seat capacity, and operating cost"),
    ("airline",         "airline",        "airline.csv",        "Airline master — name, nationality, and performance rating"),
    ("nation",          "nation",         "nation.csv",         "Nation master with parent region mapping"),
    ("region",          "region",         "region.csv",         "Geographic region master"),
]

print(f"Catalog           : {CATALOG}")
print(f"Raw volume        : {BASE_PATH}")
print(f"Pipeline run ID   : {RUN_ID}")
print(f"Tables to ingest  : {len(SOURCES)}")

In [0]:
# ── Ingestion: raw CSV → Bronze Delta ─────────────────────────────────────────
# Idempotent: every run fully overwrites the target table, so running twice
# yields the same rows as running once (only the metadata columns are refreshed).
from pyspark.sql import functions as F

row_count_results = []

for table, folder, file_name, comment in SOURCES:
    src_path = f"{BASE_PATH}/{folder}/{file_name}"
    target = f"{CATALOG}.bronze.{table}"

    # 1. Read source — header row.
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(src_path)
    )
    source_count = df.count()

    # 2. Preserve all source columns and grain; append ingestion metadata.
    bronze_df = (
        df
        .withColumn("_ingestion_timestamp", F.lit(INGESTED_AT).cast("timestamp"))
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_pipeline_run_id", F.lit(RUN_ID))
    )

    # 3. Overwrite the Bronze table (full replace — rerunnable by design).
    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    spark.sql(f"COMMENT ON TABLE {target} IS '{comment}'")

    # 4. Verify: Bronze row count must match the source row count.
    bronze_count = spark.table(target).count()
    row_count_results.append((table, file_name, source_count, bronze_count))

    status = "✓" if source_count == bronze_count else "✗ MISMATCH"
    print(f"{status} {table:<16} source={source_count:>7,}  bronze={bronze_count:>7,}")

print(f"\nIngestion complete — run ID: {RUN_ID}")

In [0]:
# ── Verification: Bronze vs. source row counts ────────────────────────────────
verification = spark.createDataFrame(
    row_count_results,
    ["bronze_table", "source_file", "source_rows", "bronze_rows"],
)
verification = verification.withColumn(
    "match", F.col("source_rows") == F.col("bronze_rows")
)
display(verification.orderBy("bronze_table"))

all_match = verification.filter(F.col("match") == False).count() == 0
print(f"\n{'✅ ALL 8 TABLES MATCH SOURCE ROW COUNTS' if all_match else '❌ ROW COUNT MISMATCHES FOUND — investigate above'}")